In [58]:
# Importing the necessary packages
import numpy as np                                  # "Scientific computing"


import pandas as pd                                 # Data Frame

import matplotlib.pyplot as plt                     # Basic visualisation

from numpy import mean
from numpy import std
from numpy import absolute
from pandas import read_csv
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

In [59]:
# Read the dataset
df = pd.read_csv('https://raw.githubusercontent.com/HOGENT-ML/course/main/datasets//clothes_size_prediction.csv')
df.head()

,weight,age,height,size
0,62,28.0,172.72,XL
1,59,36.0,167.64,L
2,61,34.0,165.10,M
3,65,27.0,175.26,L
4,62,45.0,172.72,M


## Take a look at the dataset

We'll try to predict the size based on the weight, age and height.   
  
Show some general info about the dataset

In [60]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119734 entries, 0 to 119733
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   weight  119734 non-null  int64  
 1   age     119477 non-null  float64
 2   height  119404 non-null  float64
 3   size    119734 non-null  str    
dtypes: float64(2), int64(1), str(1)
memory usage: 3.7 MB


What are number of records for each size?  

M: 29575  
S: 21829  
XXXL: 21259  
XL: 19033  
L: 17481  
XXS: 9907  
XXL: 69

In [61]:
df['size'].value_counts()

size
M       29712
S       21924
XXXL    21359
XL      19119
L       17587
XXS      9964
XXL        69
Name: count, dtype: int64

Because there are only very few records for XXL, remove those records from the dataset

In [62]:
df = df[df['size'] != 'XXL']
len(df)

119665

Train a transformer to fill in the median value of the corresponding attribute for all missing values.

In [63]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')

#df_num = df[['weight', 'age', 'height']]
df_num = df.select_dtypes(include='number')

imputer.fit(df_num)
print(imputer.statistics_)

[ 61.   32.  165.1]


Apply the imputer to the dataset and check the results.

In [64]:
df_num_tr = imputer.transform(df_num)
df_num = pd.DataFrame(df_num_tr, columns=df_num.columns, index=df_num.index)
df = pd.concat([df_num, df.drop(['weight', 'age', 'height'], axis=1)], axis=1)
df.info()

<class 'pandas.DataFrame'>
Index: 119665 entries, 0 to 119733
Data columns (total 4 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   weight  119665 non-null  float64
 1   age     119665 non-null  float64
 2   height  119665 non-null  float64
 3   size    119665 non-null  str    
dtypes: float64(3), str(1)
memory usage: 4.6 MB


At first sight this seems quite a large dataset, but is this actually true?  
First we are going to change the datatype of height from float to integer.


In [65]:

#df['height'] = (df['height'] * 100).astype('int')
df['height'] = df['height'].astype('int')
df['height'].info()
df['height'].head()

<class 'pandas.Series'>
Index: 119665 entries, 0 to 119733
Series name: height
Non-Null Count   Dtype
--------------   -----
119665 non-null  int64
dtypes: int64(1)
memory usage: 1.8 MB


0    172
1    167
2    165
3    175
4    172
Name: height, dtype: int64

It seems reasonable to round the ages to the nearest five-fold

In [66]:
df['age'] = (df['age'] / 5).round() * 5
df['age'].head(10)

0    30.0
1    35.0
2    35.0
3    25.0
4    45.0
5    25.0
6    65.0
7    35.0
8    25.0
9    30.0
Name: age, dtype: float64

Change the datatype of age from float to integer.

In [67]:
df['age'] = df['age'].astype('int')

We drop duplicate rows in the dataset.

In [68]:
df = df.drop_duplicates()

In [69]:
df.describe()

,weight,age,height
count,11330.000000,11330.000000,11330.000000
mean,64.996381,37.733010,165.483495
std,14.414304,12.694414,8.886856
min,22.000000,0.000000,137.000000
25%,55.000000,30.000000,160.000000
50%,62.000000,35.000000,165.000000
75%,72.000000,45.000000,172.000000
max,136.000000,115.000000,193.000000


In [70]:
df.info()

<class 'pandas.DataFrame'>
Index: 11330 entries, 0 to 119721
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   weight  11330 non-null  float64
 1   age     11330 non-null  int64  
 2   height  11330 non-null  int64  
 3   size    11330 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 442.6 KB


How many records are left?

In [71]:
#11330

We want to know if there are any 'wrong duplicates' in the dataset, i.e. the same values for weight, age and height, but still another size. So we count the nunique

In [72]:
wrong = df.groupby(['weight', 'age', 'height'])['size'].nunique()
num_wrong = (wrong > 1).sum()
print(num_wrong)

2726


In [73]:
#different approach
help = df.groupby(['weight', 'age', 'height']).agg('nunique').reset_index()
help

,weight,age,height,size
0,22.0,30,167,2
1,22.0,45,152,1
2,26.0,45,172,1
3,31.0,35,175,1
4,35.0,20,182,1
...,...,...,...,...
5154,136.0,30,175,1
5155,136.0,30,177,1
5156,136.0,35,165,1
5157,136.0,35,167,1


In [74]:
# Keeps first row for each (height, age, weight) group — drops the rest
df = df.drop_duplicates(subset=['height', 'age', 'weight'], keep='first')
len(df)

5159

In [75]:
# For each group → keep the row with the most common size
#df = (
#    df.groupby(['height', 'age', 'weight'], group_keys=False)
#      .apply(lambda g: g[g['size'] == g['size'].mode()[0]].iloc[0])
#      .reset_index(drop=True)
#)
#len(df)

How many records are left?

In [76]:
print(len(df))
print(df['size'].value_counts())

5159
size
XXXL    2326
M        689
XL       659
S        624
L        473
XXS      388
Name: count, dtype: int64


Check if the dataset is heavily skewed.

In [77]:
#previous cell shows this

Because we want to apply regression first, map the sizes to numbers as follows:  
'XXS' : 0, 'S' : 1, 'M': 2, 'L': 3,'XL':4,'XXXL': 5

In [78]:
df['size'] = df['size'].map({'XXS': 0, 'S': 1, 'M': 2, 'L': 3, 'XL': 4, 'XXXL': 5})
df.head()

,weight,age,height,size
0,62.0,30,172,4
1,59.0,35,167,3
2,61.0,35,165,2
3,65.0,25,175,3
4,62.0,45,172,2


What is X and what is y?

In [79]:
#X is weight, age, height
#y is size

What is X_train, y_train, X_test, y_test?

In [98]:
from sklearn.model_selection import train_test_split
X = df[['weight', 'age', 'height']]
y = df[['size']]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

What is the shape of X_train, y_train, X_test and y_test?

In [99]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(3869, 3)
(3869, 1)
(1290, 3)
(1290, 1)


What are columns of X containing text?

In [82]:
X.dtypes
#none?

weight    float64
age         int64
height      int64
dtype: object

In [102]:
#different approach
categorical_cols = X.select_dtypes(include=['object']).columns
print(categorical_cols)

Index([], dtype='str')


What are the columns of X containing numbers?

In [ ]:
X.dtypes
#all

weight    float64
age         int64
height      int64
dtype: object

In [103]:
numerical_cols = X.select_dtypes(include=['number']).columns
print(numerical_cols)

Index(['weight', 'age', 'height'], dtype='str')


Define the ColumnTransformer for applying Standard Scaling on all numeric columns.  

In [84]:
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler

X_train_num = X_train.select_dtypes('number').columns

#col_trans = ColumnTransformer(remainder='passthrough', transformers=[('std_scaler', StandardScaler(), X_train_num)])
col_trans = make_column_transformer((StandardScaler(), X_train_num), remainder='passthrough')

## Regression

Define the model LinearRegression  

In [85]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()

Define the data preparation (= ColumnTransformer for standard scaling) and modeling pipeline

In [86]:
from sklearn.pipeline import Pipeline, make_pipeline

pl = make_pipeline(col_trans, lin_reg)

Train the model

In [87]:
pl.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('columntransformer', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('standardscaler', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diffe

What is the accuracy of the model?  
Use K-fold cross-validation with k = 3.  
Find an appropriate value for the attribute scoring on [metrics and scoring](https://scikit-learn.org/stable/modules/model_evaluation.html) 

In [108]:
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score

cross_v = KFold(n_splits=3, shuffle=True, random_state=42)
scores = -cross_val_score(pl, X_train, y_train, cv=cross_v, scoring='neg_mean_squared_error')
print(scores)
print(np.mean(-cross_val_score(pl, X_train, y_train, scoring='neg_mean_squared_error', cv=3)))

[1.13924112 1.24929386 1.24636157]
1.2117481136667463


What are the values for intercept and the coefficients.  
Why do we have 3 coefficients?  
What is the most important coefficient?

In [105]:
print(lin_reg.intercept_)
print(lin_reg.coef_)
#three coefficients for the three columns
#weight seems to be the most important coefficient

[3.43397141]
[[1.33094381 0.2484815  0.00512916]]


Apply the model to the test set.  

In [90]:
y_test_pred = pl.predict(X_test)

Calculate the Mean Absolute Error and the Root Mean Squared Error

In [91]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
print(root_mean_squared_error(y_test, y_test_pred))
print(mean_absolute_error(y_test, y_test_pred))

1.103009248760387
0.9203436604478167


Interprete the results. 

## Classification

Use the softmax classifier to try to predict the class (0, 1, 2, 3, 4, 5).  
What is the accuracy score?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(C=30, random_state=42)
pl = Pipeline([('prep', col_trans), ('softmax_reg', log_reg)])

y_train_1d = np.ravel(y_train)
pl.fit(X_train, y_train_1d)

np.mean(cross_val_score(pl, X_train, y_train_1d, scoring='accuracy', cv=3))

np.float64(0.650301196969788)

Create and show the confusion matrix for the test set.

In [113]:
from sklearn.metrics import confusion_matrix

y_test_pred = pl.predict(X_test)
conf_mx = confusion_matrix(y_test, y_test_pred)
conf_mx

array([[ 46,  35,   5,   0,   0,   1],
       [ 21,  98,  41,   0,   2,   5],
       [  0,  25, 122,   0,  12,   8],
       [  1,   4,  57,   0,  35,  14],
       [  0,   3,  33,   0,  50,  70],
       [  1,   1,   8,   0,  28, 564]])

The accuracy of the classifier is low, but we see that we often predict only one size too high or too small. 
Calculate how many times 
* the classifier was correct
* the classifier predicted the size to be one size higher than the actual size
* the classifier predicted the size to be one size smaller than the actual size


In [114]:
np.diag(conf_mx).sum()

np.int64(880)

In [118]:
sum = 0
size = conf_mx.shape[0]
for i in range(1, size):
    sum += conf_mx[i - 1][i]
print(sum)
print(np.diag(conf_mx, k=1).sum())

181
181


In [119]:
sum = 0
size = conf_mx.shape[0]
for i in range(1, size):
    sum += conf_mx[i][i - 1]
print(sum)
print(np.diag(conf_mx, k=-1).sum())

131
131
